# Chapter 10: Transfer learning and global modeling

Note: This code is not executable. I have copied over the Chapter 11 source code directory so I have it to reference.

In [ ]:
import pytorch_lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class GlobalForecastingModel(L.LightningModule):
    def __init__(
    self,
    num_series: int,
    input_dim: int = 1,
    hidden_dim: int = 64,
    num_layers: int = 2,
    dropout: float = 0.1,
    learning_rate: float = 1e-3
    ):
        super().__init__()
        self.save_hyperparameters()

        # Project raw features into consistent representation
        self.feature_projection = nn.Linear(input_dim, hidden_dim)

        # Learn unique patterns for each series
        # Embedding dimension controls capacity for series-specific information
        self.series_embeddings = nn.Embedding(num_series, hidden_dim)
        self.embedding_norm = nn.LayerNorm(hidden_dim)

        # Shared temporal pattern extractor
        # All series share these LSTM parameters to learn common dynamics
        self.temporal_encoder = nn.LSTM(
        input_size=hidden_dim,
        hidden_size=hidden_dim,
        num_layers=num_layers,
        dropout=dropout if num_layers > 1 else 0,
        batch_first=True
        )

        # Combine global and series-specific patterns for prediction
        self.decoder = nn.Sequential(
        nn.Linear(hidden_dim * 2, hidden_dim),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden_dim, 1)
        )

Feature integration mechanisms


### Simple concatenation:
This is the most straightforward approach concatenates temporal features with series embeddings. This can be effective for series with similar scales and patterns.
While computationally efficient, this approach assumes equal importance of global and series-specific features.

In [ ]:
def forward(self, x, series_ids):
    # Project and encode features - shared patterns
    projected_features = self.feature_projection(x)
    temporal_features, _ = self.temporal_encoder(projected_features)

    # Get series specific patterns/embeddings
    series_embed = self.series_embeddings(series_ids)
    series_embed = self.embedding_norm(series_embed)

    # Combine them
    combined = torch.cat([temporal_features[:, -1, :], series_embed],
    dim=1)
    return self.decoder(combined)


### Attention-based integration:

For more complex relationships, attention mechanisms can dynamically weight the importance of temporal features based on series context. This makes it
particularly useful for heterogeneous series with varying temporal dependencies. We capture
the weights in our attention layer, and apply them to the temporal features.

In [ ]:
class AttentionIntegration(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.query_projection = nn.Linear(hidden_dim, hidden_dim)
        self.key_projection = nn.Linear(hidden_dim, hidden_dim)
        self.value_projection = nn.Linear(hidden_dim, hidden_dim)
        self.scale = np.sqrt(hidden_dim)

    def forward(self, temporal_features, series_embed):
        # Project series embedding to create attention query
        query = self.query_projection(
        series_embed).unsqueeze(1) # [batch, 1, dim]

        # Project temporal features to create keys and values
        keys = self.key_projection(
        temporal_features) # [batch, seq_len, dim]
        values = self.value_projection(temporal_features)

        # Compute scaled dot-product attention
        attention_scores = torch.matmul(
        query, keys.transpose(-2, -1)) / self.scale
        attention_weights = F.softmax(attention_scores, dim=-1)

        # Weight temporal features by attention
        attended_features = torch.matmul(
        attention_weights, values).squeeze(1)

        return attended_features

This mechanism allows the model to focus on relevant temporal patterns conditioned on
series-specific context, particularly valuable when series exhibit varying degrees of temporal
dependence.

### Gated integration:
Gating provides a learnable way to control the flow of information from
both global and series-specific sources through learned gates, which is useful when our series
vary in their adherence to global patterns. It does this by learning dimension-specific mixing
coefficients through an activation function, in this case sigmoid, to control the balance between
temporal and series-specific features. Unlike attention, which weights temporal relationships,
gating enables granular feature selection at each dimension; when a gate value approaches 1,
temporal features dominate, and when it approaches 0, series-specific characteristics prevail.

In [ ]:
class GatedIntegration(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.gate = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, temporal_features, series_embed):
        # Compute adaptive gates
        gate_input = torch.cat([temporal_features[:, -1, :], series_embed], dim=1)
        gates = torch.sigmoid(self.gate(gate_input))

        # Gated combination
        return gates * temporal_features[:, -1, :] + (1 - gates) * series_embed

If we were inclined, we could modify our LSTM-GFM to incorporate any of these integration mechanisms:

In [ ]:
hidden_dim = 'Placeholder for hidden dimension. Replace with actual value.'

class GlobalForecastingModel(L.LightningModule):
    def __init__(self,
                 # ...,
                 integration_type='attention'):
        super().__init__()
        # Previous initialization code...

        # Select integration mechanism
        if integration_type == 'attention':
            self.integration = AttentionIntegration(hidden_dim)
        elif integration_type == 'gated':
            self.integration = GatedIntegration(hidden_dim)
        else:
            self.integration = None # Use simple concatenation


### Data organization and batching

Designing data structures for GFMs demands balancing computational efficiency with effective modeling performance. We need to handle multiple series while preserving temporal coherence and
memory efficiency. Our PyTorch implementation utilizes index management and sliding windows to
efficiently access data and preserve temporal sequences.

In [ ]:
# import warnings
import warnings

class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, series_dict, lookback, horizon, min_samples=100):
        self.lookback = lookback
        self.horizon = horizon
        self.samples = []

        # Create mapping from series ID to numerical index
        self.series_to_idx = {sid: idx for idx, sid in enumerate(series_dict.keys())}

        # Preprocess and validate each series
        for series_id, data in series_dict.items():
            if len(data) < lookback + horizon:
                warnings.warn(f"Series {series_id} too short, skipping")
                continue

            series_idx = self.series_to_idx[series_id]
            valid_windows = self._generate_windows(data, series_idx)
            self.samples.extend(valid_windows)

        if len(self.samples) < min_samples:
            raise ValueError("Insufficient valid samples for training")

        def _generate_windows(self, data, series_idx):
            windows = []
            for t in range(len(data) - self.lookback - self.horizon + 1):
                windows.append({
                 'series_idx': series_idx,
                 'temporal_idx': t,
                 'input': self._normalize_window(data[t:t+self.lookback]),
                 'target': data[t+self.lookback:t+self.lookback+self.horizon]
                })
            return windows

        def _normalize_window(self, window):
            # Add scale-invariant normalization
            mean = window.mean()
            std = window.std()
            return (window - mean) / (std + 1e-8)


### Training and balanced sampling
Training a GFM requires only a slight change from single-series training: batching through series. We
wrap this into a configure_training function that wires together the data loaders, balanced sampling,
and trainer configuration.

In [ ]:
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping


def configure_training(model, train_dataset, val_dataset, **kwargs):
    # Configure data loaders with balanced sampling
    train_sampler = create_balanced_sampler(train_dataset)
    train_loader = DataLoader(train_dataset,
     batch_size=kwargs.get('batch_size', 32),
     sampler=train_sampler,
     num_workers=kwargs.get('num_workers', 4)
     )

    val_loader = DataLoader(
    val_dataset,
     batch_size=kwargs.get('batch_size', 32),
     shuffle=False,
     num_workers=kwargs.get('num_workers', 4)
    )

    # Configure trainer with early stopping
    trainer = pl.Trainer(
    max_epochs=kwargs.get('max_epochs', 100),
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    gradient_clip_val=0.1,
    callbacks=[EarlyStopping(
         monitor='val_loss',
         patience=kwargs.get('patience', 10),
         mode='min')]
    )

    return trainer, train_loader, val_loader

This implementation is deceptively simple, masking some underlying complexity; each batch has to
maintain coherent temporal windows while balancing representation across different series. As with
batching in general, there are advantages and disadvantages to the impact of this on computational
efficiency and statistical robustness.

The architectures we’ve explored provide a foundation for knowledge transfer across series, but their
effectiveness depends heavily on how we expose the model to different patterns during training. Default sampling approaches can create subtle biases that limit a model’s ability to learn transferable
patterns, particularly when series vary in length or characteristics. If we consider how sampling works
in practice: when generating training batches, we typically take windows from all available series and
sample randomly from this pool. An issue here is that a series with more observations contributes
more windows to our sampling pool than a short series. Given two series:

In [ ]:
# Series lengths
series_A = 1000 # ~970 possible windows
series_B = 100 # ~70 possible windows

# Probability of sampling from each in random batch
p_series_A = 970 / (970 + 70) # ≈ 93%
p_series_B = 70 / (970 + 70) # ≈ 7%

Our model would see patterns from longer series far more frequently, creating a training bias that
undermines transfer learning. The solution is weighted sampling that balances representation across
series:

In [ ]:
from collections import Counter
from torch.utils.data.sampler import WeightedRandomSampler

def create_balanced_sampler(dataset):
    """Create sampler that evenly weights series representation"""
    # Calculate series frequencies
    series_counts = Counter(sample['series_idx'] for sample in dataset.samples)
    # Inverse frequency weighting
    weights = torch.zeros(len(dataset))
    for idx, sample in enumerate(dataset.samples):
        weights[idx] = 1.0 / series_counts[sample['series_idx']]

    # Normalize weights
    weights = weights / weights.sum()

    return WeightedRandomSampler(
     weights=weights,
     num_samples=len(dataset),
     replacement=True
    )

We apply this in our DataLoader, like so.

In [ ]:
train_dataset = "placeholder for train_dataset. Replace with actual value."

train_loader = DataLoader(
 train_dataset,
 batch_size=32,
 sampler=create_balanced_sampler(train_dataset),
 num_workers=4
)

Under this weighting scheme each series has equal probability of appearing in our training batches,
regardless of length. The impact of balanced sampling extends beyond simple fairness; it fundamentally affects what patterns our model learns and how well they transfer, providing more diverse
pattern exposure during training, better gradient updates for underrepresented series, improved
generalization across varying series lengths, and reduced overfitting to dominant patterns. Our experiments show this balanced approach typically improves transfer performance, though training
may take longer as the model sees more varied patterns. The sampling approach can extend beyond
series length to balance any relevant characteristic, be it magnitude of values, frequency of events,
or other domain-specific features.

Note: The companion notebooks include additional diagnostic utilities (analyze_sampling_
distribution, check_batch_bias) for examining how different series contribute to
your training batches and verifying that your sampling strategy is working as intended.

### Optimizers, learning rate scheduling, and batching

Learning rate scheduling proves particularly crucial for global models due to the complex loss landscape created by optimizing across multiple series simultaneously. We use an adaptive learning rate
schedule through PyTorch’s ReduceLROnPlateau scheduler, alongside the Adam optimizer (introduced
in Chapter 6):

In [2]:
def configure_optimizers(self):
    optimizer = torch.optim.Adam(self.parameters(),
    lr=self.hparams.learning_rate)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',patience=5,factor=0.5)

    return {
     'optimizer': optimizer,
     'lr_scheduler': {
     'scheduler': scheduler,
     'monitor': 'val_loss'
    }}

The scheduler monitors validation loss and reduces the learning rate when performance plateaus. This
adaptation proves essential as the optimizer must navigate a loss surface that combines both shared
and series-specific patterns. The patience parameter allows the model to explore potentially promising
regions before reducing the learning rate, while the factor controls the magnitude of adjustments.

Batch processing fundamentally influences both computational efficiency and model learning dynamics. Our experiments suggest optimal batch sizes typically range between 32-128 series windows,
though this varies with dataset characteristics. We default to 32 in our examples as a conservative
starting point that works well for moderately sized datasets; larger batches expedite training through
increased parallelization, yet risk averaging out important temporal dependencies. For substantial
datasets, memory constraints may necessitate streaming approaches to batch generation, where windows are grouped by series to preserve temporal ordering within each series while randomizing series
order between epochs. Multiple workers facilitate efficient data loading through parallel processing,
particularly valuable when handling extensive panel datasets; however, you must ensure that the data
loading process preserves the temporal integrity of each series.

Validation demands particular attention in global modeling contexts. Our approach incorporates both
temporal and transfer validation, assessing performance on recent data from known series as well
as entirely held-out series.

In [ ]:
def validation_step(self, batch, batch_idx):
    y_hat = self.forward(batch['input'], batch['series_ids'])
    loss = F.mse_loss(y_hat, batch['target'])

    self.log('val_loss', loss)
    self._log_series_metrics(y_hat, batch)

    return loss

Once we have set up our network and optimizers, it is simply a case of structuring our code. The following example is the general structure you should follow for initializing and training a GFM model.

In [ ]:
# See source repo for full code for this chapter.
series_dict = {"Adding this so code compiles. Replace with actual value."}
feature_dim = "Adding this so code compiles. Replace with actual value."
train_series = "Adding this so code compiles. Replace with actual value."


# Model initialization
model = GlobalForecastingModel(
 num_series=len(series_dict),
 input_dim=feature_dim,
 hidden_dim=64
)

# Data preparation
train_dataset = TimeSeriesDataset(
 series_dict=train_series,
 lookback=24,
 horizon=12
)

# Training configuration
trainer = pl.Trainer(
 max_epochs=100,
 accelerator='gpu' if torch.cuda.is_available() else 'cpu',
 devices=1,
 gradient_clip_val=0.1
)

# Execute training
trainer.fit(
 model,
 DataLoader(
 train_dataset,
    batch_size=32,
 sampler=create_balanced_sampler(train_dataset),
 num_workers=4
 )
)

PyTorch Lightning API change :The gpus=1 parameter in PyTorch Lightning’s Trainer is deprecated. Current versions
use accelerator='gpu', devices=1 (or accelerator='cpu' for CPU training). The
code examples throughout this chapter use the updated API, but if you are following older
tutorials or documentation you may encounter the legacy syntax.

The modifications for converting standard architectures to GFMs are fairly straightforward to implement while maintaining the core adaptability of the base architecture and enabling cross-series learning.
Most modern deep learning frameworks support these adaptations with minimal code changes. The
complexity lies not in the modifications themselves, but in selecting integration mechanisms and
embedding dimensions based on the characteristics of your time series ensemble. For example, concatenation suits homogeneous series, while attention mechanisms may better serve heterogeneous
patterns.

### Transfer learning with NeuralForecast


For this section, I would reference the book or source code. Not everything is provided. Book makes it easier to follow. Not hard. Just not bothering to document fully.

Big picture of this section is we compare statistical, ML (tree based), GFM models, and pretraining GFM models prior to fine-tuning.

#### Data preparation

In [ ]:
import pandas as pd
air_train = pd.concat(air_train, axis=0)
air_test = pd.concat(air_test, axis=0)
air_test.head(n=3)

#### Statistical baselines

In [ ]:
from statsforecast.models import SeasonalNaive, AutoARIMA, AutoETS, Theta
from statsforecast import StatsForecast

fh = 18

models = [
 SeasonalNaive(season_length=12),
 AutoARIMA(),
 AutoETS(),
 Theta()
]

sf = StatsForecast(
 models=models,
 freq='ME',
 n_jobs=-1
)
sf.fit(df=air_train)

This code creates a single model for each series in the dataset, for each model type. We then take the
fitted object and create forecasts by simply passing the forecast horizon length (h=fh) to the .forecast
method. This creates a dataframe containing the predictions, with the predictions for each model in
a separate column, which we can join to the actuals from the test set (air_test) to create a dataframe
for evaluating model performance (eval_df).

In [ ]:
fcst = sf.forecast(h=fh)
fcst = fcst.reset_index()

eval_df = fcst.merge(
 air_test[['unique_id', 'ds', 'y']],
 on=['unique_id', 'ds'],
 how='left'
)

Evaluation is simply a case of running the evaluate function on your forecast results. This function
computes the performance of our model(s) using a set of specified metrics from utilsforecast.
losses, such as Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), etc. We pass these
metrics, model names (matching column names), standard columns (id_col, time_col, target_col),
and the training data (air_train) which is used to create metrics like MASE that require a baseline
(e.g., a naive forecast) for scaling.

In [ ]:
from functools import partial
from utilsforecast.evaluation import evaluate
from Utilities.ErrorMetricsUtilities import mae, mse, rmse, mase, mape, smape

series_metrics = evaluate(
 eval_df,
models=[model_name], # Change to match column name
 metrics=metrics,
 train_df=air_train,
 id_col='unique_id',
 time_col='ds',
 target_col='y'
)

At this point, we have built individual models for each statistical modeling algorithm. We take the
median results for each metric and algorithm and put them in a table for our comparison. However,
it is wise to look at and statistically compare the distribution of errors across the different metrics
(generally RMSE, MAE and some relative metric).

You can see below that AutoARIMA tends to perform better on most metrics, although it is beaten by
the Seasonal Naive on both MAE and MASE (<1 is better than naive). We could make the argument that
some feature engineering could enhance the AutoARIMA, but overall these models are not impressing
greatly beyond a naive. They are, however, all extremely quick to build.

### Decision-tree global models

In [ ]:
from mlforecast import MLForecast
from mlforecast.lag_transforms import ExpandingMean, ExponentiallyWeightedMean,
RollingMean
import lightgbm as lgb
from sklearn.linear_model import LinearRegression
from utilsforecast.feature_engineering import time_features

# Train Linear Regression
linear_model = MLForecast(
 models={'linear': LinearRegression()},
 freq='ME',
 lags=range(1,12),
 date_features=['month', 'year']
)
linear_model.fit(
 df=air_train,
 id_col='unique_id',
 time_col='ds',
 target_col='y',
 static_features=[]
)
linear_preds = linear_model.predict(h=fh)

# Train LightGBM
lgb_model = MLForecast(
 models=[lgb.LGBMRegressor(verbosity=-1,random_state=352, objective='mae')],
 freq='ME',
 lags=range(1,12),
 lag_transforms={
 1: [RollingMean(12), ExponentiallyWeightedMean(alpha=0.3)],
 12: [ExpandingMean()]
 },
 date_features=['month', 'year']
)
lgb_model.fit(air_train, id_col='unique_id', time_col='ds', target_col='y',
static_features=[])
lgb_preds = lgb_model.predict(h=fh)

### Neural network GFMs

In [ ]:
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NHITS
# NBEATS
nbeats_config = NBEATS(
 h=18,
 input_size= fh * 2, # sum=36
 n_blocks=[1]*20,
 mlp_units=[[256, 256]] *3,
 stack_types=['identity', 'trend', 'seasonality'],
 learning_rate=1e-3,
 batch_size=1024,
 random_seed=243,
 max_steps=10,
 val_check_steps=100,
 scaler_type='standard'
)

# Initialize NeuralForecast with NBEATS
nbeats_model = NeuralForecast(
 models=[nbeats_config],
     freq='ME'
)
# Train NBEATS
nbeats_model.fit(df=air_train)

In [ ]:
# Generate NBEATS forecasts
nbeats_preds = nbeats_model.predict(futr_df=air_test)

### Pretraining with external data

In [ ]:
# Set-up
nbeats_model = NeuralForecast(
 models=[nbeats_config],
 freq='ME'
)
# Pretrain with M4 data
nbeats_model.fit(df=m4_train)
# Generate forecasts for air_test
nbeats_preds = nbeats_model.predict(df=air_train, futr_df=air_test)